# Modelo baseline para churn voluntário

Este notebook tem como objetivo construir um primeiro modelo interpretável para estimar o risco de churn voluntário nos próximos 90 dias.

A modelagem usa dados sintéticos e deve ser lida como uma demonstração técnica. Os resultados não representam desempenho real em produção.

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH_CANDIDATES = [
    Path("data/customers_churn_synthetic.csv"),
    Path("../data/customers_churn_synthetic.csv"),
]

DATA_PATH = next(path for path in DATA_PATH_CANDIDATES if path.exists())

df = pd.read_csv(DATA_PATH)

df.head()


,customer_id,tenure_months,access_technology,download_speed_mbps,monthly_fee,has_contract_loyalty,overdue_invoice_count,oldest_overdue_days,active_agreement_installment_amount,had_price_adjustment_90d,support_tickets_90d,repeat_issue_90d,avg_resolution_hours_90d,outage_count_30d,network_outage_hours_30d,churn_90d
0,CUST-000001,40,fiber,500,160.93,0,0,0,0.00,1,0,0,0.00,0,0.00,0
1,CUST-000002,53,fiber,200,109.99,1,0,0,0.00,0,0,0,0.00,0,0.00,0
2,CUST-000003,35,fiber,100,100.73,0,1,46,0.00,0,0,0,0.00,0,0.00,0
3,CUST-000004,31,fiber,200,116.91,1,0,0,61.27,0,0,0,0.00,1,1.13,1
4,CUST-000005,58,fiber,100,105.92,1,2,32,0.00,0,1,0,47.23,0,0.00,0


## Checagem inicial e definição do alvo

Antes de treinar qualquer modelo, vamos confirmar a estrutura da base e separar o alvo `churn_90d` das variáveis de entrada. Essa separação evita incluir o próprio resultado que queremos prever como informação para o modelo.


In [2]:
print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")
print(f"Taxa de churn voluntário: {df['churn_90d'].mean():.2%}")

TARGET = "churn_90d"
FEATURES = [column for column in df.columns if column not in ["customer_id", TARGET]]

print(f"Variáveis de entrada: {len(FEATURES)}")
FEATURES


Linhas: 5000
Colunas: 16
Taxa de churn voluntário: 10.34%
Variáveis de entrada: 14


['tenure_months',
 'access_technology',
 'download_speed_mbps',
 'monthly_fee',
 'has_contract_loyalty',
 'overdue_invoice_count',
 'oldest_overdue_days',
 'active_agreement_installment_amount',
 'had_price_adjustment_90d',
 'support_tickets_90d',
 'repeat_issue_90d',
 'avg_resolution_hours_90d',
 'outage_count_30d',
 'network_outage_hours_30d']

## Separação entre treino e teste

Vamos separar a base em dois conjuntos: treino e teste. O conjunto de treino será usado para ajustar o modelo, e o conjunto de teste ficará reservado para avaliar o desempenho em dados que o modelo ainda não viu.

A separação será estratificada pelo alvo `churn_90d`, para manter uma proporção parecida de churn nos dois conjuntos.


In [3]:
from sklearn.model_selection import train_test_split

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "base": ["treino", "teste"],
        "linhas": [len(X_train), len(X_test)],
        "taxa_churn": [y_train.mean(), y_test.mean()],
    }
)
split_summary["taxa_churn"] = split_summary["taxa_churn"].mul(100).round(2)

split_summary


,base,linhas,taxa_churn
0,treino,3750,10.35
1,teste,1250,10.32


## Pré-processamento das variáveis

A regressão logística precisa receber apenas números. Por isso, vamos separar variáveis numéricas e categóricas.

As variáveis numéricas serão padronizadas para ficarem em escala comparável. A variável categórica `access_technology` será convertida em colunas indicadoras com one-hot encoding.


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = ["access_technology"]
numeric_features = [column for column in FEATURES if column not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ]
)

print(f"Variáveis numéricas: {len(numeric_features)}")
print(f"Variáveis categóricas: {len(categorical_features)}")

numeric_features, categorical_features


Variáveis numéricas: 13
Variáveis categóricas: 1


(['tenure_months',
  'download_speed_mbps',
  'monthly_fee',
  'has_contract_loyalty',
  'overdue_invoice_count',
  'oldest_overdue_days',
  'active_agreement_installment_amount',
  'had_price_adjustment_90d',
  'support_tickets_90d',
  'repeat_issue_90d',
  'avg_resolution_hours_90d',
  'outage_count_30d',
  'network_outage_hours_30d'],
 ['access_technology'])

## Treino do modelo baseline

O primeiro modelo será uma regressão logística. Ela é uma boa escolha inicial porque é simples, interpretável e produz probabilidades de churn voluntário.

Nesta etapa, o objetivo não é buscar o melhor modelo possível, mas criar uma referência confiável para comparar melhorias futuras.


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

model.fit(X_train, y_train)

print("Modelo treinado com sucesso.")


Modelo treinado com sucesso.


## Avaliação inicial do modelo

Agora vamos medir o desempenho no conjunto de teste, que representa clientes que o modelo não usou durante o treino.

Como churn é um evento minoritário, não basta olhar apenas a acurácia. Também vamos observar precision, recall, F1, ROC-AUC e PR-AUC.


In [6]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics_summary = pd.DataFrame(
    {
        "métrica": [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
        ],
        "valor": [
            accuracy_score(y_test, y_pred),
            precision_score(y_test, y_pred),
            recall_score(y_test, y_pred),
            f1_score(y_test, y_pred),
            roc_auc_score(y_test, y_proba),
            average_precision_score(y_test, y_proba),
        ],
    }
)

metrics_summary["valor"] = metrics_summary["valor"].round(4)

metrics_summary

,métrica,valor
0,accuracy,0.6184
1,precision,0.1345
2,recall,0.4961
3,f1,0.2116
4,roc_auc,0.6069
5,pr_auc,0.1808
